# Labels

`labels=True` (the default) draws the whole style, `labels=False` drops its labels and icons (its symbol layers), and `labels="only"` draws only them, on a transparent background (grey here, dark for a dark style). A city, a region and a country:

In [ ]:
%pip install -q basemaplibre geopandas

In [ ]:
import geopandas
import matplotlib.pyplot as plt

import basemaplibre as bml

In [ ]:
def around(lon, lat, half_width):
    """Axes limits of a square around a point, half_width Web Mercator meters to each side."""
    x, y = geopandas.GeoSeries.from_xy([lon], [lat], crs=4326).to_crs(3857).iloc[0].coords[0]
    return x - half_width, x + half_width, y - half_width, y + half_width

In [ ]:
warsaw = around(21.013, 52.237, 1500)
examples = {
    "OpenFreeMap.Liberty": (warsaw, "0.8"),
    "CartoDB.Voyager": (around(135.4, 34.75, 60000), "0.8"),  # Kansai: Osaka, Kobe and Kyoto
    "OpenFreeMap.Dark": (around(12.5, 42.1, 800000), "0.15"),  # Italy
}
choices = [True, False, "only"]
fig, axs = plt.subplots(3, 3, figsize=(8.4, 8.8), layout="constrained")
for ax, labels in zip(axs[0], choices, strict=True):
    ax.set_title(f"labels={labels!r}")
for row, (name, (extent, background)) in zip(axs, examples.items(), strict=True):
    row[0].set_ylabel(name)
    for ax, labels in zip(row, choices, strict=True):
        ax.axis(extent)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_facecolor(background)
        bml.add_basemap(ax, source=name, labels=labels)

## Data between the map and its labels

`labels="above"` draws the map under the data and its labels above it: here, a choropleth of GDP per capita. The labels go above the data and below the legend, at `labels_zorder` (4 by default), as those of `labels="only"` do.

In [ ]:
NE = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/"
countries = geopandas.read_file(NE + "ne_110m_admin_0_countries.geojson")
europe = (
    countries[countries.CONTINENT == "Europe"]
    .to_crs(3035)
    .assign(gdp=lambda df: df.GDP_MD / df.POP_EST * 1e6)
)

legend = {"label": "GDP per capita (USD)", "shrink": 0.7}
ax = europe.plot(
    "gdp",
    figsize=(6.4, 5.6),
    cmap="YlGn",
    alpha=0.8,
    edgecolor="white",
    legend=True,
    legend_kwds=legend,
)
ax.axis((2.5e6, 6.5e6, 1.4e6, 5.2e6))
bml.add_basemap(ax, crs=europe.crs, source="CartoDB.Voyager", labels="above")